# Create Sample Player Characters

We'll define a number of features of a Character.

1. Attributes and Skills.

    Work with a starting budget of 18D for attributes and 7D for skills.
    Each attribute (except Extranormal) must be in the range $1D \leq a \leq 5D$.
    Skills are constained of $0 \leq s \leq 3D$.

    Extranormal means sometimes there are 6 and other times 7 attributes.

2. Advantages, Disadvantages and Special Abilities.  These are filled in with "TBD" everywhere because the details aren't easy to generate. The player needs to work out some descriptions that create a complete character.
3. Weapons, Armor, and Equipment.

    These come from a campaign book or OpenD6 Fantasy rulebook.
    The source ``.csv`` files are ingested to get weapons, armor, shields, and equipment.

In [1]:
from collections import Counter, defaultdict
from contextlib import redirect_stdout
import csv
from pathlib import Path
from random import randint, randrange, shuffle, choice
import re
from statistics import variance
import subprocess
import textwrap
from typing import Self

In [2]:
from opend6_tools.character import *

## Core Features: Attributes and Skills

There are three pools of attributes, the common attributes everyone shares.
Then there are two specialized pools for mages and clerics.

In [3]:
common_attributes = [Agility, Intellect, Coordination, Acumen, Physique, Charisma]
mage_attributes = common_attributes + [Magic]
cleric_attributes = common_attributes + [Miracles]

Allocate the attributes. There are two candidate approaches. This implementation uses the second.

1. Use a ``Counter``. The total dice, $18D$, based on $D=3$, is 54 pips.
   Generate 54 numbers from 1-6 or 1-7. A ``Counter`` of these numbers will provide a distribution of pips among the attributes. It's often very flat.

2. Divide the pips equitably among the various attributes. (For 6, this is 3D each.)
   Iterate the following: pick a pair of attributes and a deflection amount.
   If one attribute can be deflected down and the other attribute deflected up, then adjust the budget.
   If the variance is high, or this has been done 6 times, stop the iteration.

In [4]:
def allocate_attributes(d, domain, low, high) -> list[tuple[Attribute, DieCode]]:
    """
    Allocate d dice among the given domain of attributes (common or common + extranormal),
    with the constraint given by low and high.
    """
    pips = int(d.measure)
    pips_min, pips_max = low.measure, high.measure
    base, extra = divmod(pips, len(domain))
    budget = {a: base for a in range(len(domain))}
    for _ in range(extra):
        budget[randrange(len(domain))] += 1

    # Deflect pairs: one up another down, until the variance is big.
    # It seems like 6 attemps is enough, it may touch everything twice.
    keys = list(budget.keys())
    for _ in range(6):
        deflection = randrange(1, 7)
        shuffle(keys)
        up, down = keys[:2]
        if (pips_min <= budget[up] + deflection <= pips_max) and (pips_min <= budget[down] - deflection <= pips_max):
            budget[up] += deflection
            budget[down] -= deflection
        # print(deflection, up, down, budget, variance(budget.values()))
        if variance(budget.values()) > 6:
            # Stop early, this was most wonderfully asymmetrical
            break

    attrs = list(domain)
    shuffle(attrs)
    return [
        (domain[c], D.from_pips(budget[c]))
        for c in budget.keys()
    ]

allocate_attributes(18*D, common_attributes, 1*D, 5*D)

[(opend6_tools.character.features.Agility, 2*D+2),
 (opend6_tools.character.features.Intellect, 2*D+2),
 (opend6_tools.character.features.Coordination, 3*D),
 (opend6_tools.character.features.Acumen, 2*D+1),
 (opend6_tools.character.features.Physique, 4*D+1),
 (opend6_tools.character.features.Charisma, 3*D)]

Allocate some skills. First, pick a subset of skills.
Then allocate points using an inverse power curve to spread points widely, then add to an ever-narrowing collection of skills.

In [5]:
def allocate_skills(d, domain, low, high) -> list[tuple[tuple[Attribute, str], DieCode]]:
    skills = [
        (a, s)
        for a in domain
        for s in a.SKILL_NAMES
    ]
    available = int(d.measure)
    assert low.measure == 0
    assert high.measure <= 2**9, f"algorithm doesn't work for {high}"
    shuffle(skills)
    subset = skills[:available // 2]
    skill_budget = Counter()
    while available:
        shuffle(subset)
        pool = subset[:available // 2 or 1]
        skill_budget += Counter(pool)
        available -= available // 2 or 1
    return list(skill_budget.items())

allocate_skills(7*D, common_attributes, 0*D, 3*D)

[((opend6_tools.character.features.Charisma, 'mettle'), 2),
 ((opend6_tools.character.features.Charisma, 'animal handling'), 1),
 ((opend6_tools.character.features.Charisma, 'charm'), 4),
 ((opend6_tools.character.features.Intellect, 'devices'), 2),
 ((opend6_tools.character.features.Agility, 'dodge'), 1),
 ((opend6_tools.character.features.Physique, 'lifting'), 3),
 ((opend6_tools.character.features.Acumen, 'disguise'), 3),
 ((opend6_tools.character.features.Agility, 'stealth'), 2),
 ((opend6_tools.character.features.Coordination, 'lockpicking'), 1),
 ((opend6_tools.character.features.Agility, 'contortion'), 2)]

## Bonus Features: Advantages, Disadvantages, and Special Abilities

The Advantage, Disadvantage, SpecialAbility triple is another interesting budget-balancing issue.

Some characters of 0 to 2 advantages, 0 to 1 special abilities (which vary in cost.)
These are balanced by disadvantages.

While this is available in features.ADVANTAGE_RULES, features.SPECIALABILITY_RULES, and features.DISADVANTAGE_RULES, these are not exported normally. Instead the direct subclasses will do: ``Advantage.__subclasses__()``, etc.

In [6]:
def allocate_special():
    adv_count = choice([0] * 6 + [1] * 4)  # From template characters.
    spec_count = choice([0] * 3 + [1] * 6 + [2])
    advantages = [choice(Advantage.__subclasses__()) for _ in range(adv_count)]
    special_abilities = [choice(SpecialAbility.__subclasses__()) for _ in range(spec_count)]
    dis_count = adv_count + sum(s.rank_cost for s in special_abilities)
    disadvantages = [choice(Disadvantage.__subclasses__()) for _ in range(dis_count)]
    return advantages, disadvantages, special_abilities

## Armor, Weapons, Equipment

First, read the given CSV source files. This needs to be cached to facilitate generating multiple characters.

Armor: 7/10, 1 item. Shield: 1/10, 1 item.

Weapon: 10/10. 1 item. Missile 1/10 vs. Melee 9/10.

Gear: 8/10, 1-4 items (decreasing in probability)

In [7]:
class CampaignCache:
    melee: list[str]
    missile: list[str]
    armor: list[str]
    shield: list[str]
    gear: list[str]
    def __init__(self) -> None:
        here = Path.cwd()
        self.fantasy_rules_path = here.parent.parent / "fantasy_rulebook"
    def load(self, base_path: Path | None = None) -> Self:
        rules_path = base_path or self.fantasy_rules_path
        footnote_pattern = re.compile(r"^([^\[]*)(\[.*?\]_)?$")
        with open(rules_path/"melee.csv") as source_file:
            raw_melee = list(
                m['Type'].strip()
                for m in csv.DictReader(source_file, fieldnames=['Type', 'DMG', 'Price'], dialect=csv.Dialect.skipinitialspace)
                if m['Type'] and m['Type'].strip() != 'Type'
            )
            self.melee = list(
                footnote_pattern.match(m.strip()).group(1)
                for m in raw_melee
            )
        with open(rules_path/"missiles.csv") as source_file:
            self.missile = list(
                m['Type'].lstrip('-').strip()
                for m in csv.DictReader(source_file, fieldnames=['Type', 'Damage', 'S', 'M', 'L', 'Price'], dialect=csv.Dialect.skipinitialspace)
                if m['Damage'] and m['Type'] and m['Type'].strip() != 'Type'
            )
        with open(rules_path/"armor.csv") as source_file:
            self.armor = list(
                a['Type'].strip()
                for a in csv.DictReader(source_file, fieldnames=['Type', 'AV', 'Price'], dialect=csv.Dialect.skipinitialspace)
                if a['Type'] and a['Type'].strip() != 'Type' and a['Price'].strip().startswith('M')  # Avoid expensive plate armor
            )
        with open(rules_path/"shields.csv") as source_file:
            self.shield = list(
                s['Type'].strip()
                for s in csv.DictReader(source_file, fieldnames=['Type', 'AV', 'Price'], dialect=csv.Dialect.skipinitialspace)
                if s['Type'] and s['Type'].strip() != 'Type'
            )
        with open(rules_path/"gear.csv") as source_file:
            self.gear = list(
                g['Item']
                for g in csv.DictReader(source_file, fieldnames=['Item', 'Price'], dialect=csv.Dialect.skipinitialspace)
                if g['Item'] and g['Item'].strip() != 'Item'
            )
        return self

    def __call__(self):
        n_armor = choice([0] * 3 + [1] * 7)
        n_shield = choice([0] * 9 + [1])
        weapon = choice([self.missile] + [self.melee]*10)
        n_gear = choice([4]*4 + [3]*3 + [2]*2 + [1]*1 + [0]*2)
        return {
            'armor': (
                [choice(self.armor) for _ in range(n_armor)]
                + [choice(self.shield) for _ in range(n_shield)]
            ),
            'weapons': [choice(weapon)],
            'equipment': [choice(self.gear) for _ in range(n_gear)],
        }

In [8]:
allocate_equipment = CampaignCache().load()
equip_map = allocate_equipment()
equip_map

{'armor': ['Bone and hide'],
 'weapons': ['Sap, hammer (tool)'],
 'equipment': ['Bell, small metal', 'Grappling hook', 'Whetstone']}

## Build the character

In [9]:
def create_character(alloc_eq: CampaignCache, attributes: list[Attribute] | None = None):
    """
    ..  todo:: Add budgets and limits.
    """
    def attr_parameter_name(attr_class: type[Attribute]) -> str:
        if issubclass(attr_class, (Magic, Miracles)):
            return "extranormal"
        return attr_class.__name__.lower()

    if not attributes:
        attributes = choice(
            8*[common_attributes] + [mage_attributes, cleric_attributes]
        )
    attr_values = allocate_attributes(18*D, attributes, 1*D, 5*D)
    skill_values = allocate_skills(7*D, attributes, 0*D, 3*D)
    advantages, disadvantages, special_abilities = allocate_special()

    skill_map = defaultdict(dict)
    for (attr, skill), value in skill_values:
        skill_map[attr][skill] = value
    attr_map = {
        attr_parameter_name(attr_class): attr_class(value, skill_map[attr_class])
        for attr_class, value in attr_values
    }

    spec_map = {
        'advantages': [a(1, "TBD") for a in advantages],
        'disadvantages': [d(1, "TBD") for d in disadvantages],
        'special_abilities': [s(1, "TBD") for s in special_abilities],
    }
    equip_map = alloc_eq()

    character = Character(
        name="",
        occupation="",
        gender = choice(["male", "male", "female", "female", "trans", "inter"]),
        height=None,  # To force computation
        weight=None,
        **attr_map,
        **spec_map,
        **equip_map,
    )
    return character

character = create_character(allocate_equipment, common_attributes)
print(display(character))

{'name': '',
 'occupation': '',
 'race': '',
 'gender': 'female',
 'age': '',
 'height': '170cm',
 'weight': '90kg',
 'physical_description': '',
 'agility': Agility(3*D, {'riding': 1*D, 'jumping': 0*D+1, 'fighting': 0*D+1, 'dodge': 0*D+2}),
 'intellect': Intellect(4*D+2, {'speaking': 1*D}),
 'coordination': Coordination(1*D+1, {'throwing': 1*D}),
 'acumen': Acumen(3*D+1, {'know-how': 1*D, 'artist': 0*D+2, 'crafting': 0*D+2}),
 'physique': Physique(3*D+1, {'lifting': 0*D, 'running': 0*D, 'stamina': 0*D, 'swimming': 0*D}),
 'charisma': Charisma(2*D+1, {'command': 0*D+1}),
 'extranormal': Magic(0*D, {'alteration': 0*D, 'apportation': 0*D, 'conjuration': 0*D, 'divination': 0*D}),
 'advantages': [],
 'disadvantages': [],
 'special_abilities': [],
 'description': '',
 'realm': 'Human realm',
 'move': 10,
 'strength_damage': 2*D,
 'body': 28,
 'wounds': {'Mortal': '1-2',
            'Incapacitated': '3-5',
            'Severe': '6-10',
            'Wounded': '11-16',
            'Stunned': '

In [10]:
eval(repr(character))

Character(name='', occupation='', race='', gender='female', age='', height='170cm', weight='90kg', physical_description='', agility=Agility(3*D, {'riding': 1*D, 'jumping': 0*D+1, 'fighting': 0*D+1, 'dodge': 0*D+2}), intellect=Intellect(4*D+2, {'speaking': 1*D}), coordination=Coordination(1*D+1, {'throwing': 1*D}), acumen=Acumen(3*D+1, {'know-how': 1*D, 'artist': 0*D+2, 'crafting': 0*D+2}), physique=Physique(3*D+1, {'lifting': 0*D, 'running': 0*D, 'stamina': 0*D, 'swimming': 0*D}), charisma=Charisma(2*D+1, {'command': 0*D+1}), extranormal=Magic(0*D, {'alteration': 0*D, 'apportation': 0*D, 'conjuration': 0*D, 'divination': 0*D}), advantages=[], disadvantages=[], special_abilities=[], description='', realm='Human realm', move=10, strength_damage=2*D, body=28, wounds={'Mortal': '1-2', 'Incapacitated': '3-5', 'Severe': '6-10', 'Wounded': '11-16', 'Stunned': '17-21'}, funds=4*D, silver=240, fate_points=1, character_points=5, equipment=NoteList('Mirror, silver', 'Quiver', 'Sealing wax', 'Seal

## One-shot Character Builder

This creates a character, reports in LaTeX and then runs **pdflatex** to create the PDF.

In [11]:
here = Path.cwd()
forms = here.parent / "forms"
sample_path = forms / "sample.tex"
print(f"Creating {sample_path}")
sample_path.write_text(Format.LATEX.value().report(character))
subprocess.run(["pdflatex", str(sample_path), f"-output-directory={forms}"], cwd=forms, check=True)

Creating /Users/slott/Documents/Hobbies/Gaming/OpenD6/world/world_book/forms/sample.tex
This is pdfTeX, Version 3.141592653-2.6-1.40.29 (TeX Live 2026) (preloaded format=pdflatex)
 restricted \write18 enabled.
entering extended mode

(/Users/slott/Documents/Hobbies/Gaming/OpenD6/world/world_book/forms/sample.tex
LaTeX2e <2026-06-01>
L3 programming layer <2026-05-26>
(/usr/local/texlive/2026/texmf-dist/tex/latex/base/article.cls
Document Class: article 2025/01/22 v1.4n Standard LaTeX document class
(/usr/local/texlive/2026/texmf-dist/tex/latex/base/size10.clo))
(/usr/local/texlive/2026/texmf-dist/tex/latex/amsmath/amsmath.sty
For additional information on amsmath, use the `?' option.
(/usr/local/texlive/2026/texmf-dist/tex/latex/amsmath/amstext.sty
(/usr/local/texlive/2026/texmf-dist/tex/latex/amsmath/amsgen.sty))
(/usr/local/texlive/2026/texmf-dist/tex/latex/amsmath/amsbsy.sty)
(/usr/local/texlive/2026/texmf-dist/tex/latex/amsmath/amsopn.sty))
(/usr/local/texlive/2026/texmf-dist/tex/la

CompletedProcess(args=['pdflatex', '/Users/slott/Documents/Hobbies/Gaming/OpenD6/world/world_book/forms/sample.tex', '-output-directory=/Users/slott/Documents/Hobbies/Gaming/OpenD6/world/world_book/forms'], returncode=0)

## Template Characters

It's often helpful to create a batch of template characters.

This works out well as follows:

1. Create a ``.py`` module with a batch of characters, perhaps in a ``forms`` directory.

2. Run ``ruff format`` on the module.

2. Run the ``.py`` module to emit the LaTeX files into the ``forms`` directory.

3. Run **pdflatex** to create the required PDF files from the LaTeX files.

In [14]:
with open(forms / "template.py", "w") as template_file:
    with redirect_stdout(template_file):
        print('''"""Template Characters"""''')
        print("from opend6_tools.character import *")

        mix = [common_attributes]*8 + [mage_attributes, cleric_attributes]
        for n, attr_mix in enumerate(mix, start=1):
            char = create_character(allocate_equipment, attr_mix)
            print(f"c_{n} = {repr(char)}")

        print("all_characters = {")
        for n in range(len(mix)):
            print(f"    'c_{n+1}': c_{n+1},")
        print("}")

        print(textwrap.dedent("""\
        if __name__ == "__main__":
            app = build_app(all_characters)
            app()
        """))

In [18]:
subprocess.run(["/Users/slott/.local/bin/uvx", "ruff", "format", str(forms / "template.py")], check=True)
for n in range(len(mix)):
    subprocess.run(["python", "template.py", "pdf", "--format=latex", f"c_{n+1}"], cwd=forms, check=True)

1 file left unchanged
Creating /Users/slott/Documents/Hobbies/Gaming/OpenD6/world/world_book/forms/c_1.tex
pdflatex /Users/slott/Documents/Hobbies/Gaming/OpenD6/world/world_book/forms/c_1.tex
This is pdfTeX, Version 3.141592653-2.6-1.40.29 (TeX Live 2026) (preloaded format=pdflatex)
 restricted \write18 enabled.
entering extended mode
(/Users/slott/Documents/Hobbies/Gaming/OpenD6/world/world_book/forms/c_1.tex
LaTeX2e <2026-06-01>
L3 programming layer <2026-05-26>
(/usr/local/texlive/2026/texmf-dist/tex/latex/base/article.cls
Document Class: article 2025/01/22 v1.4n Standard LaTeX document class
(/usr/local/texlive/2026/texmf-dist/tex/latex/base/size10.clo))
(/usr/local/texlive/2026/texmf-dist/tex/latex/amsmath/amsmath.sty
For additional information on amsmath, use the `?' option.
(/usr/local/texlive/2026/texmf-dist/tex/latex/amsmath/amstext.sty
(/usr/local/texlive/2026/texmf-dist/tex/latex/amsmath/amsgen.sty))
(/usr/local/texlive/2026/texmf-dist/tex/latex/amsmath/amsbsy.sty)
(/usr/loc